## Setup and Imports

In [7]:
# Standard library imports
import sys
from pathlib import Path
from datetime import datetime, timedelta

# Third-party imports
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project imports
from ayne.utils.query_utils import execute_custom_query

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 80)

print(f"✓ Setup complete - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Setup complete - 2025-11-29 11:44:30


## 1. Refresh Strategy Validation

Verify that refresh flags are being set correctly and freezing logic is working as intended.

### 1.1 Refresh Flag Consistency

In [8]:
# Check if last_full_refresh is set correctly
query = """
SELECT
    COUNT(*) as total_movies,
    COUNT(CASE WHEN last_tmdb_update IS NOT NULL THEN 1 END) as has_tmdb,
    COUNT(CASE WHEN last_omdb_update IS NOT NULL THEN 1 END) as has_omdb,
    COUNT(CASE WHEN last_tmdb_update IS NOT NULL AND last_omdb_update IS NOT NULL THEN 1 END) as has_both,
    COUNT(CASE WHEN last_full_refresh IS NOT NULL THEN 1 END) as has_full_refresh,
    COUNT(CASE WHEN last_tmdb_update IS NOT NULL AND last_omdb_update IS NOT NULL AND last_full_refresh IS NULL THEN 1 END) as missing_full_refresh
FROM movies
"""

refresh_flags = execute_custom_query(query)

print("\n" + "="*70)
print(" "*20 + "REFRESH FLAG AUDIT")
print("="*70)

total = refresh_flags['total_movies'].iloc[0]
has_tmdb = refresh_flags['has_tmdb'].iloc[0]
has_omdb = refresh_flags['has_omdb'].iloc[0]
has_both = refresh_flags['has_both'].iloc[0]
has_full = refresh_flags['has_full_refresh'].iloc[0]
missing_full = refresh_flags['missing_full_refresh'].iloc[0]

print(f"\n📊 Total Movies: {total:,}")
print(f"\n🎬 TMDB Updates: {has_tmdb:,} ({100*has_tmdb/total:.1f}%)")
print(f"🎭 OMDB Updates: {has_omdb:,} ({100*has_omdb/total:.1f}%)")
print(f"✅ Both APIs Updated: {has_both:,} ({100*has_both/total:.1f}%)")
print(f"🔄 Full Refresh Flag Set: {has_full:,} ({100*has_full/total:.1f}%)")

if missing_full > 0:
    print(f"\n⚠️  ISSUE: {missing_full:,} movies have both updates but missing last_full_refresh!")
    print(f"   Expected: {has_both:,} | Actual: {has_full:,} | Gap: {missing_full:,}")
else:
    print(f"\n✅ All movies with both updates have last_full_refresh set correctly")

print("="*70)


                    REFRESH FLAG AUDIT

📊 Total Movies: 32,534

🎬 TMDB Updates: 32,534 (100.0%)
🎭 OMDB Updates: 3,783 (11.6%)
✅ Both APIs Updated: 3,783 (11.6%)
🔄 Full Refresh Flag Set: 3,783 (11.6%)

✅ All movies with both updates have last_full_refresh set correctly


### 1.2 Freezing Logic Validation

In [9]:
# Check frozen status by movie age category
query = """
SELECT
    CASE
        WHEN DATEDIFF('day', release_date, CURRENT_DATE) <= 60 THEN 'Recent (0-60 days)'
        WHEN DATEDIFF('day', release_date, CURRENT_DATE) <= 180 THEN 'Established (60-180 days)'
        WHEN DATEDIFF('day', release_date, CURRENT_DATE) <= 365 THEN 'Mature (180-365 days)'
        ELSE 'Archived (>365 days)'
    END as age_category,
    COUNT(*) as total,
    COUNT(CASE WHEN data_frozen = TRUE THEN 1 END) as frozen,
    ROUND(100.0 * COUNT(CASE WHEN data_frozen = TRUE THEN 1 END) / COUNT(*), 1) as frozen_pct,
    AVG(consecutive_unchanged_refreshes) as avg_unchanged_cycles,
    MAX(consecutive_unchanged_refreshes) as max_unchanged_cycles
FROM movies
GROUP BY age_category
ORDER BY
    CASE age_category
        WHEN 'Recent (0-60 days)' THEN 1
        WHEN 'Established (60-180 days)' THEN 2
        WHEN 'Mature (180-365 days)' THEN 3
        ELSE 4
    END
"""

frozen_by_age = execute_custom_query(query)

print("\n" + "="*80)
print(" "*25 + "FREEZING LOGIC AUDIT")
print("="*80)
print("\n📋 Frozen Status by Movie Age:\n")
print(frozen_by_age.to_string(index=False))

# Check for anomalies
recent_frozen = frozen_by_age[frozen_by_age['age_category'].str.contains('Recent')]['frozen'].sum()
established_frozen = frozen_by_age[frozen_by_age['age_category'].str.contains('Established')]['frozen'].sum()
mature_frozen = frozen_by_age[frozen_by_age['age_category'].str.contains('Mature')]['frozen'].sum()

issues = []
if recent_frozen > 0:
    issues.append(f"⚠️  {recent_frozen} Recent movies are frozen (should be 0)")
if established_frozen > 0:
    issues.append(f"⚠️  {established_frozen} Established movies are frozen (should be 0)")
if mature_frozen > 0:
    issues.append(f"⚠️  {mature_frozen} Mature movies are frozen (should be 0)")

if issues:
    print("\n⚠️  ISSUES DETECTED:")
    for issue in issues:
        print(f"   {issue}")
    print("\n   ℹ️  Only Archived movies (>365 days) should be frozen, and only after 3+ unchanged cycles")
else:
    print("\n✅ Freezing logic working correctly - only archived movies are frozen")

print("="*80)


                         FREEZING LOGIC AUDIT

📋 Frozen Status by Movie Age:

             age_category  total  frozen  frozen_pct  avg_unchanged_cycles  max_unchanged_cycles
       Recent (0-60 days)     42       0         0.0                   0.0                     0
Established (60-180 days)    127       0         0.0                   0.0                     0
    Mature (180-365 days)    291       0         0.0                   0.0                     0
     Archived (>365 days)  32074       0         0.0                   0.0                     0

✅ Freezing logic working correctly - only archived movies are frozen


### 1.3 Consecutive Unchanged Cycles Tracking

In [10]:
# Analyze consecutive unchanged cycles distribution
query = """
SELECT
    consecutive_unchanged_refreshes as cycles,
    COUNT(*) as movie_count,
    COUNT(CASE WHEN data_frozen = TRUE THEN 1 END) as frozen_count
FROM movies
GROUP BY consecutive_unchanged_refreshes
ORDER BY consecutive_unchanged_refreshes
"""

cycle_dist = execute_custom_query(query)

print("\n📊 Distribution of Consecutive Unchanged Cycles:\n")
print(cycle_dist.to_string(index=False))

# Check if movies are being frozen at correct threshold
frozen_below_threshold = cycle_dist[(cycle_dist['cycles'] < 3) & (cycle_dist['frozen_count'] > 0)]

if not frozen_below_threshold.empty:
    total_wrong = frozen_below_threshold['frozen_count'].sum()
    print(f"\n⚠️  ISSUE: {total_wrong} movies frozen with < 3 unchanged cycles!")
    print("   These should not be frozen yet.")
else:
    print("\n✅ All frozen movies have 3+ consecutive unchanged cycles")

# Visualize distribution
if len(cycle_dist) > 1:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=cycle_dist['cycles'],
        y=cycle_dist['movie_count'],
        name='Total Movies',
        marker_color='lightblue'
    ))

    fig.add_trace(go.Bar(
        x=cycle_dist['cycles'],
        y=cycle_dist['frozen_count'],
        name='Frozen Movies',
        marker_color='#ff6b6b'
    ))

    fig.add_vline(x=3, line_dash="dash", line_color="green",
                  annotation_text="Freeze Threshold", annotation_position="top right")

    fig.update_layout(
        title='Distribution of Consecutive Unchanged Cycles',
        xaxis_title='Consecutive Unchanged Cycles',
        yaxis_title='Number of Movies',
        barmode='overlay',
        height=450
    )

    fig.show()


📊 Distribution of Consecutive Unchanged Cycles:

 cycles  movie_count  frozen_count
      0        32534             0

✅ All frozen movies have 3+ consecutive unchanged cycles


## 2. Data Completeness Audit

Check for missing or incomplete data across critical fields.

### 2.1 Core Movie Fields

In [11]:
# Check completeness of core fields
query = """
SELECT
    COUNT(*) as total,
    ROUND(100.0 * COUNT(CASE WHEN title IS NOT NULL AND title != '' THEN 1 END) / COUNT(*), 2) as title_pct,
    ROUND(100.0 * COUNT(CASE WHEN release_date IS NOT NULL THEN 1 END) / COUNT(*), 2) as release_date_pct,
    ROUND(100.0 * COUNT(CASE WHEN tmdb_id IS NOT NULL THEN 1 END) / COUNT(*), 2) as tmdb_id_pct,
    ROUND(100.0 * COUNT(CASE WHEN imdb_id IS NOT NULL AND imdb_id != '' THEN 1 END) / COUNT(*), 2) as imdb_id_pct,
    ROUND(100.0 * COUNT(CASE WHEN last_tmdb_update IS NOT NULL THEN 1 END) / COUNT(*), 2) as tmdb_update_pct,
    ROUND(100.0 * COUNT(CASE WHEN last_omdb_update IS NOT NULL THEN 1 END) / COUNT(*), 2) as omdb_update_pct
FROM movies
"""

completeness = execute_custom_query(query)

print("\n" + "="*70)
print(" "*20 + "DATA COMPLETENESS AUDIT")
print("="*70)
print("\n📊 Core Movie Fields:\n")

total = completeness['total'].iloc[0]
for col in completeness.columns:
    if col == 'total':
        print(f"Total Movies: {completeness[col].iloc[0]:,}\n")
    elif col.endswith('_pct'):
        field_name = col.replace('_pct', '').replace('_', ' ').title()
        value = completeness[col].iloc[0]
        status = "✅" if value >= 95 else "⚠️" if value >= 80 else "❌"
        print(f"{status} {field_name:20s}: {value:6.2f}%")

print("="*70)


                    DATA COMPLETENESS AUDIT

📊 Core Movie Fields:

Total Movies: 32,534

✅ Title               : 100.00%
✅ Release Date        : 100.00%
✅ Tmdb Id             : 100.00%
✅ Imdb Id             :  99.11%
✅ Tmdb Update         : 100.00%
❌ Omdb Update         :  11.63%


### 2.2 TMDB Enrichment Fields

In [12]:
# Check TMDB enrichment completeness
query = """
SELECT
    COUNT(*) as total,
    ROUND(100.0 * COUNT(CASE WHEN overview IS NOT NULL AND overview != '' THEN 1 END) / COUNT(*), 2) as overview_pct,
    ROUND(100.0 * COUNT(CASE WHEN genres IS NOT NULL AND genres != '' THEN 1 END) / COUNT(*), 2) as genres_pct,
    ROUND(100.0 * COUNT(CASE WHEN production_companies IS NOT NULL AND production_companies != '' THEN 1 END) / COUNT(*), 2) as companies_pct,
    ROUND(100.0 * COUNT(CASE WHEN production_countries IS NOT NULL AND production_countries != '' THEN 1 END) / COUNT(*), 2) as countries_pct,
    ROUND(100.0 * COUNT(CASE WHEN runtime IS NOT NULL AND runtime > 0 THEN 1 END) / COUNT(*), 2) as runtime_pct,
    ROUND(100.0 * COUNT(CASE WHEN vote_count IS NOT NULL AND vote_count > 0 THEN 1 END) / COUNT(*), 2) as votes_pct,
    ROUND(100.0 * COUNT(CASE WHEN spoken_languages IS NOT NULL AND spoken_languages != '' THEN 1 END) / COUNT(*), 2) as languages_pct
FROM tmdb_movies
"""

tmdb_completeness = execute_custom_query(query)

print("\n📊 TMDB Enrichment Fields:\n")

total_tmdb = tmdb_completeness['total'].iloc[0]
print(f"Total TMDB Records: {total_tmdb:,}\n")

for col in tmdb_completeness.columns:
    if col != 'total' and col.endswith('_pct'):
        field_name = col.replace('_pct', '').replace('_', ' ').title()
        value = tmdb_completeness[col].iloc[0]
        status = "✅" if value >= 90 else "⚠️" if value >= 70 else "❌"
        print(f"{status} {field_name:20s}: {value:6.2f}%")


📊 TMDB Enrichment Fields:

Total TMDB Records: 32,534

✅ Overview            :  99.80%
✅ Genres              :  99.94%
✅ Companies           :  97.42%
✅ Countries           :  99.24%
✅ Runtime             :  99.83%
✅ Votes               : 100.00%
✅ Languages           :  99.73%


### 2.3 OMDB Enrichment Fields

In [13]:
# Check OMDB enrichment completeness
query = """
SELECT
    COUNT(*) as total,
    ROUND(100.0 * COUNT(CASE WHEN rated IS NOT NULL AND rated != '' AND rated != 'N/A' THEN 1 END) / COUNT(*), 2) as rated_pct,
    ROUND(100.0 * COUNT(CASE WHEN imdb_rating IS NOT NULL AND imdb_rating > 0 THEN 1 END) / COUNT(*), 2) as imdb_rating_pct,
    ROUND(100.0 * COUNT(CASE WHEN imdb_votes IS NOT NULL AND imdb_votes > 0 THEN 1 END) / COUNT(*), 2) as imdb_votes_pct,
    ROUND(100.0 * COUNT(CASE WHEN metascore IS NOT NULL AND metascore > 0 THEN 1 END) / COUNT(*), 2) as metascore_pct,
    ROUND(100.0 * COUNT(CASE WHEN awards IS NOT NULL AND awards != '' AND awards != 'N/A' THEN 1 END) / COUNT(*), 2) as awards_pct,
    ROUND(100.0 * COUNT(CASE WHEN box_office IS NOT NULL AND box_office > 0 THEN 1 END) / COUNT(*), 2) as box_office_pct,
    ROUND(100.0 * COUNT(CASE WHEN language IS NOT NULL AND language != '' AND language != 'N/A' THEN 1 END) / COUNT(*), 2) as language_pct
FROM omdb_movies
"""

omdb_completeness = execute_custom_query(query)

print("\n📊 OMDB Enrichment Fields:\n")

total_omdb = omdb_completeness['total'].iloc[0]
print(f"Total OMDB Records: {total_omdb:,}\n")

for col in omdb_completeness.columns:
    if col != 'total' and col.endswith('_pct'):
        field_name = col.replace('_pct', '').replace('_', ' ').title()
        value = omdb_completeness[col].iloc[0]
        status = "✅" if value >= 80 else "⚠️" if value >= 50 else "❌"
        print(f"{status} {field_name:20s}: {value:6.2f}%")


📊 OMDB Enrichment Fields:

Total OMDB Records: 3,875

⚠️ Rated               :  63.35%
✅ Imdb Rating         :  93.37%
✅ Imdb Votes          :  98.76%
❌ Metascore           :  43.46%
⚠️ Awards              :  60.75%
❌ Box Office          :  26.92%
✅ Language            :  98.55%

Total OMDB Records: 3,875

⚠️ Rated               :  63.35%
✅ Imdb Rating         :  93.37%
✅ Imdb Votes          :  98.76%
❌ Metascore           :  43.46%
⚠️ Awards              :  60.75%
❌ Box Office          :  26.92%
✅ Language            :  98.55%


## 3. Timestamp Consistency Audit

Verify that update timestamps are consistent across related tables.

In [14]:
# Check for timestamp inconsistencies
query = """
SELECT
    COUNT(*) as total_movies,
    COUNT(CASE WHEN m.last_tmdb_update IS NOT NULL AND t.tmdb_id IS NULL THEN 1 END) as tmdb_timestamp_orphans,
    COUNT(CASE WHEN m.last_omdb_update IS NOT NULL AND o.imdb_id IS NULL THEN 1 END) as omdb_timestamp_orphans,
    COUNT(CASE WHEN t.tmdb_id IS NOT NULL AND m.last_tmdb_update IS NULL THEN 1 END) as tmdb_data_without_timestamp,
    COUNT(CASE WHEN o.imdb_id IS NOT NULL AND m.last_omdb_update IS NULL THEN 1 END) as omdb_data_without_timestamp
FROM movies m
LEFT JOIN tmdb_movies t ON m.tmdb_id = t.tmdb_id
LEFT JOIN omdb_movies o ON m.imdb_id = o.imdb_id
"""

timestamp_audit = execute_custom_query(query)

print("\n" + "="*70)
print(" "*15 + "TIMESTAMP CONSISTENCY AUDIT")
print("="*70)

total = timestamp_audit['total_movies'].iloc[0]
tmdb_orphans = timestamp_audit['tmdb_timestamp_orphans'].iloc[0]
omdb_orphans = timestamp_audit['omdb_timestamp_orphans'].iloc[0]
tmdb_missing = timestamp_audit['tmdb_data_without_timestamp'].iloc[0]
omdb_missing = timestamp_audit['omdb_data_without_timestamp'].iloc[0]

print(f"\n📊 Total Movies: {total:,}\n")

issues_found = False

if tmdb_orphans > 0:
    print(f"❌ {tmdb_orphans:,} movies have last_tmdb_update but no TMDB data")
    issues_found = True

if omdb_orphans > 0:
    print(f"❌ {omdb_orphans:,} movies have last_omdb_update but no OMDB data")
    issues_found = True

if tmdb_missing > 0:
    print(f"⚠️  {tmdb_missing:,} movies have TMDB data but last_tmdb_update is NULL")
    issues_found = True

if omdb_missing > 0:
    print(f"⚠️  {omdb_missing:,} movies have OMDB data but last_omdb_update is NULL")
    issues_found = True

if not issues_found:
    print("✅ All timestamps are consistent with corresponding data tables")

print("="*70)


               TIMESTAMP CONSISTENCY AUDIT

📊 Total Movies: 32,534

✅ All timestamps are consistent with corresponding data tables

               TIMESTAMP CONSISTENCY AUDIT

📊 Total Movies: 32,534

✅ All timestamps are consistent with corresponding data tables


## 4. API Enrichment Coverage by Year

Analyze enrichment patterns across release years to identify gaps.

In [15]:
# Get enrichment coverage by year (last 20 years)
query = """
SELECT
    EXTRACT(YEAR FROM release_date) as release_year,
    COUNT(*) as total_movies,
    COUNT(CASE WHEN last_tmdb_update IS NOT NULL THEN 1 END) as tmdb_enriched,
    COUNT(CASE WHEN last_omdb_update IS NOT NULL THEN 1 END) as omdb_enriched,
    COUNT(CASE WHEN last_full_refresh IS NOT NULL THEN 1 END) as fully_refreshed,
    ROUND(100.0 * COUNT(CASE WHEN last_tmdb_update IS NOT NULL THEN 1 END) / COUNT(*), 1) as tmdb_pct,
    ROUND(100.0 * COUNT(CASE WHEN last_omdb_update IS NOT NULL THEN 1 END) / COUNT(*), 1) as omdb_pct,
    ROUND(100.0 * COUNT(CASE WHEN last_full_refresh IS NOT NULL THEN 1 END) / COUNT(*), 1) as full_pct
FROM movies
WHERE EXTRACT(YEAR FROM release_date) >= EXTRACT(YEAR FROM CURRENT_DATE) - 20
GROUP BY release_year
ORDER BY release_year DESC
"""

coverage_by_year = execute_custom_query(query)

print("\n📅 API Enrichment Coverage (Last 20 Years):\n")
print(coverage_by_year.to_string(index=False))

# Identify years with low coverage
low_omdb = coverage_by_year[coverage_by_year['omdb_pct'] < 50]
if not low_omdb.empty:
    print(f"\n⚠️  Years with <50% OMDB coverage:")
    for _, row in low_omdb.iterrows():
        print(f"   {int(row['release_year'])}: {row['omdb_pct']:.1f}% ({row['omdb_enriched']}/{row['total_movies']})")

# Visualize coverage trends
if len(coverage_by_year) > 0:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=coverage_by_year['release_year'],
        y=coverage_by_year['tmdb_pct'],
        name='TMDB %',
        mode='lines+markers',
        line=dict(color='#01b4e4', width=2)
    ))

    fig.add_trace(go.Scatter(
        x=coverage_by_year['release_year'],
        y=coverage_by_year['omdb_pct'],
        name='OMDB %',
        mode='lines+markers',
        line=dict(color='#f5c518', width=2)
    ))

    fig.add_trace(go.Scatter(
        x=coverage_by_year['release_year'],
        y=coverage_by_year['full_pct'],
        name='Full Refresh %',
        mode='lines+markers',
        line=dict(color='#10b981', width=2)
    ))

    fig.add_hline(y=90, line_dash="dash", line_color="green", annotation_text="Target: 90%")

    fig.update_layout(
        title='API Enrichment Coverage by Year',
        xaxis_title='Release Year',
        yaxis_title='Coverage %',
        hovermode='x unified',
        height=500,
        yaxis_range=[0, 105]
    )

    fig.show()


📅 API Enrichment Coverage (Last 20 Years):

 release_year  total_movies  tmdb_enriched  omdb_enriched  fully_refreshed  tmdb_pct  omdb_pct  full_pct
         2025           397            397            258              258     100.0      65.0      65.0
         2024           791            791            714              714     100.0      90.3      90.3
         2023           984            984            978              978     100.0      99.4      99.4
         2022          1083           1083           1073             1073     100.0      99.1      99.1
         2021          1092           1092            760              760     100.0      69.6      69.6
         2020          1038           1038              0                0     100.0       0.0       0.0
         2019          1353           1353              0                0     100.0       0.0       0.0
         2018          1390           1390              0                0     100.0       0.0       0.0
         2

## 5. Data Quality Outliers

Identify suspicious or anomalous data that may need manual review.

### 5.1 Movies with Suspicious Ratings

In [16]:
# Find movies with extreme or suspicious ratings
query = """
SELECT
    m.movie_id,
    m.title,
    m.release_date,
    t.vote_average,
    t.vote_count,
    o.imdb_rating,
    o.imdb_votes,
    ABS(COALESCE(t.vote_average, 0) - COALESCE(o.imdb_rating, 0)) as rating_diff
FROM movies m
LEFT JOIN tmdb_movies t ON m.tmdb_id = t.tmdb_id
LEFT JOIN omdb_movies o ON m.imdb_id = o.imdb_id
WHERE (
    (t.vote_average IS NOT NULL AND o.imdb_rating IS NOT NULL
     AND ABS(t.vote_average - o.imdb_rating) > 2.5)  -- Large discrepancy
    OR (t.vote_average IS NOT NULL AND t.vote_average > 9.5)  -- Suspiciously high
    OR (o.imdb_rating IS NOT NULL AND o.imdb_rating > 9.5)  -- Suspiciously high
)
ORDER BY rating_diff DESC
LIMIT 20
"""

suspicious_ratings = execute_custom_query(query)

if not suspicious_ratings.empty:
    print("\n⚠️  Movies with Suspicious Ratings:\n")
    print(suspicious_ratings.to_string(index=False))
    print("\n   ℹ️  Large discrepancies (>2.5 points) may indicate data quality issues")
else:
    print("\n✅ No suspicious rating discrepancies found")


⚠️  Movies with Suspicious Ratings:

 movie_id                                   title release_date  vote_average  vote_count  imdb_rating  imdb_votes  rating_diff
    30574                               Kill Shot   2023-08-15         8.324         204          3.3        1977        5.024
    29456                                Succubus   2024-10-06         9.000         188          4.6        1520        4.400
    30325                    The Way to the Heart   2024-04-02         9.853         143          5.7         118        4.153
    36025   Harry and Meghan: Escaping the Palace   2021-09-06         6.700         156          2.6        1421        4.100
    32772                    Thor: God of Thunder   2022-07-08         6.246          57          2.3         469        3.946
    30824                  4 Horsemen: Apocalypse   2022-04-29         6.037         123          2.1         482        3.937
    28708                           Night Carnage   2025-07-29         5.

### 5.2 Movies with Missing Critical Fields

In [17]:
# Find movies missing critical fields despite being enriched
query = """
SELECT
    m.movie_id,
    m.title,
    m.release_date,
    CASE WHEN t.overview IS NULL OR t.overview = '' THEN '❌' ELSE '✅' END as overview,
    CASE WHEN t.genres IS NULL OR t.genres = '' THEN '❌' ELSE '✅' END as genres,
    CASE WHEN t.runtime IS NULL OR t.runtime = 0 THEN '❌' ELSE '✅' END as runtime,
    CASE WHEN o.imdb_rating IS NULL OR o.imdb_rating = 0 THEN '❌' ELSE '✅' END as imdb_rating,
    m.last_tmdb_update,
    m.last_omdb_update
FROM movies m
LEFT JOIN tmdb_movies t ON m.tmdb_id = t.tmdb_id
LEFT JOIN omdb_movies o ON m.imdb_id = o.imdb_id
WHERE m.last_full_refresh IS NOT NULL  -- Should have complete data
  AND (
      t.overview IS NULL OR t.overview = ''
      OR t.genres IS NULL OR t.genres = ''
      OR t.runtime IS NULL OR t.runtime = 0
      OR (o.imdb_id IS NOT NULL AND (o.imdb_rating IS NULL OR o.imdb_rating = 0))
  )
ORDER BY m.release_date DESC
LIMIT 30
"""

missing_critical = execute_custom_query(query)

if not missing_critical.empty:
    print("\n⚠️  Fully Refreshed Movies with Missing Critical Fields:\n")
    print(missing_critical.to_string(index=False))
    print(f"\n   Found {len(missing_critical)} movies that may need manual review")
else:
    print("\n✅ All fully refreshed movies have complete critical fields")


⚠️  Fully Refreshed Movies with Missing Critical Fields:

 movie_id                                      title release_date overview genres runtime imdb_rating           last_tmdb_update           last_omdb_update
    28639                           Wicked: For Good   2025-11-16        ✅      ✅       ✅           ❌ 2025-11-29 09:30:58.653401 2025-11-29 09:31:49.285758
    28640              Now You See Me: Now You Don't   2025-11-12        ✅      ✅       ✅           ❌ 2025-11-29 09:30:58.653401 2025-11-29 09:31:49.285758
    28681                      A Merry Little Ex-Mas   2025-11-12        ✅      ✅       ✅           ❌ 2025-11-29 09:30:58.653401 2025-11-29 09:31:49.285758
    28631                          The Family Plan 2   2025-11-11        ✅      ✅       ✅           ❌ 2025-11-29 09:30:58.653401 2025-11-29 09:31:49.285758
    30243                               The Stranger   2025-10-29        ✅      ✅       ✅           ❌ 2025-11-23 17:20:05.419076 2025-11-25 22:08:00.681917
    3

## 6. Audit Summary and Recommendations

Generate an overall health score and prioritized action items.

In [18]:
# Calculate overall health score
print("\n" + "="*70)
print(" "*20 + "DATABASE HEALTH SUMMARY")
print("="*70)

# Gather key metrics
metrics = {}

# Refresh flags
if 'refresh_flags' in locals():
    metrics['refresh_consistency'] = 100 if refresh_flags['missing_full_refresh'].iloc[0] == 0 else \
        max(0, 100 - (refresh_flags['missing_full_refresh'].iloc[0] / refresh_flags['has_both'].iloc[0] * 100))

# Freezing logic
if 'frozen_by_age' in locals():
    wrong_frozen = frozen_by_age[~frozen_by_age['age_category'].str.contains('Archived')]['frozen'].sum()
    total = frozen_by_age['total'].sum()
    metrics['freezing_accuracy'] = max(0, 100 - (wrong_frozen / total * 100))

# Data completeness (average of core fields > 95%)
if 'completeness' in locals():
    core_fields = [completeness[col].iloc[0] for col in completeness.columns if col.endswith('_pct')]
    metrics['data_completeness'] = sum(core_fields) / len(core_fields)

# Timestamp consistency
if 'timestamp_audit' in locals():
    issues = (timestamp_audit['tmdb_timestamp_orphans'].iloc[0] +
              timestamp_audit['omdb_timestamp_orphans'].iloc[0] +
              timestamp_audit['tmdb_data_without_timestamp'].iloc[0] +
              timestamp_audit['omdb_data_without_timestamp'].iloc[0])
    metrics['timestamp_consistency'] = max(0, 100 - (issues / timestamp_audit['total_movies'].iloc[0] * 100))

# Calculate overall score
if metrics:
    overall_score = sum(metrics.values()) / len(metrics)

    print(f"\n🎯 Overall Health Score: {overall_score:.1f}/100\n")

    for metric_name, score in metrics.items():
        label = metric_name.replace('_', ' ').title()
        status = "✅" if score >= 95 else "⚠️" if score >= 80 else "❌"
        print(f"{status} {label:25s}: {score:5.1f}/100")

    print("\n" + "="*70)

    # Generate recommendations
    print("\n📋 RECOMMENDED ACTIONS:\n")

    recommendations = []

    if metrics.get('refresh_consistency', 100) < 95:
        recommendations.append("🔧 Run migration to backfill missing last_full_refresh flags")

    if metrics.get('freezing_accuracy', 100) < 95:
        recommendations.append("🔧 Review and unfreeze movies that don't meet freezing criteria")

    if metrics.get('data_completeness', 100) < 90:
        recommendations.append("📊 Prioritize enrichment for movies with incomplete core fields")

    if metrics.get('timestamp_consistency', 100) < 95:
        recommendations.append("🕐 Investigate and fix timestamp/data mismatches")

    if 'coverage_by_year' in locals():
        low_coverage = coverage_by_year[coverage_by_year['omdb_pct'] < 70]
        if not low_coverage.empty:
            recommendations.append(f"📅 Focus OMDB enrichment on {len(low_coverage)} years with <70% coverage")

    if recommendations:
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec}")
    else:
        print("✅ No critical issues found - database is healthy!")

    print("\n" + "="*70)

print(f"\n🕐 Audit completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


                    DATABASE HEALTH SUMMARY

🎯 Overall Health Score: 96.3/100

✅ Refresh Consistency      : 100.0/100
✅ Freezing Accuracy        : 100.0/100
⚠️ Data Completeness        :  85.1/100
✅ Timestamp Consistency    : 100.0/100


📋 RECOMMENDED ACTIONS:

1. 📊 Prioritize enrichment for movies with incomplete core fields
2. 📅 Focus OMDB enrichment on 18 years with <70% coverage


🕐 Audit completed at 2025-11-29 11:44:32


## 7. Custom Audit Queries

Space for ad-hoc audit queries as needed.

In [19]:
# Example: Check for duplicate entries
query = """
SELECT
    tmdb_id,
    COUNT(*) as duplicate_count,
    STRING_AGG(movie_id::VARCHAR, ', ') as movie_ids
FROM movies
WHERE tmdb_id IS NOT NULL
GROUP BY tmdb_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
"""

duplicates = execute_custom_query(query)

if not duplicates.empty:
    print("\n⚠️  Duplicate TMDB IDs Found:\n")
    print(duplicates.to_string(index=False))
else:
    print("\n✅ No duplicate TMDB IDs found")


✅ No duplicate TMDB IDs found


In [20]:
# Example: Find movies with impossible dates (e.g., updated before created)
query = """
SELECT
    movie_id,
    title,
    created_at,
    last_tmdb_update,
    last_omdb_update,
    last_full_refresh
FROM movies
WHERE (last_tmdb_update < created_at)
   OR (last_omdb_update < created_at)
   OR (last_full_refresh < created_at)
LIMIT 20
"""

invalid_dates = execute_custom_query(query)

if not invalid_dates.empty:
    print("\n⚠️  Movies with Invalid Timestamps:\n")
    print(invalid_dates.to_string(index=False))
else:
    print("\n✅ No invalid timestamp sequences found")


⚠️  Movies with Invalid Timestamps:

 movie_id                                          title                 created_at           last_tmdb_update last_omdb_update last_full_refresh
    28633                       One Battle After Another 2025-11-23 18:23:08.083808 2025-11-23 16:23:08.028088              NaT               NaT
    28638                              War of the Worlds 2025-11-23 18:23:08.083808 2025-11-23 16:23:08.028088              NaT               NaT
    28643 Demon Slayer: Kimetsu no Yaiba Infinity Castle 2025-11-23 18:23:08.083808 2025-11-23 16:23:08.028088              NaT               NaT
    28645             Chainsaw Man - The Movie: Reze Arc 2025-11-23 18:23:08.083808 2025-11-23 16:23:08.028088              NaT               NaT
    28648                             KPop Demon Hunters 2025-11-23 18:23:08.083808 2025-11-23 16:23:08.028088              NaT               NaT
    28651                      The Conjuring: Last Rites 2025-11-23 18:23:08.083808 20

---

## Usage Notes

**Run this audit notebook regularly to:**
- Verify refresh strategy is working correctly
- Identify data quality issues early
- Monitor API enrichment progress
- Catch anomalies before they become problems

**Recommended Schedule:**
- After major data collection runs
- Weekly for active development
- Monthly for production monitoring

**Key Health Indicators:**
- Overall health score should be >90
- Refresh consistency should be 100%
- Data completeness should be >95% for core fields
- No frozen movies <365 days old

*Last run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*